In [2]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca")


In [6]:
def format_alpaca(example):
    instruction = example['instruction']
    input_text = example['input']
    output = example['output']

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    return {
        "prompt": prompt,
        "response": output
    }

formatted_dataset = dataset['train'].map(format_alpaca)


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-6.9b")
tokenizer.pad_token = tokenizer.eos_token  # Set pad token to eos

# Tokenize a single string (prompt + response) for causal LM
def tokenize_function(example):
    full_text = example['prompt'] + example['response']
    tokenized = tokenizer(
        full_text,
        truncation=True,
        padding='max_length',
        max_length=512,
        return_tensors=None  # Ensure it's a dict, not tensor objects
    )
    return tokenized

# Make sure batched=True and remove_columns to avoid shape errors
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names,
)


In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
import torch

model_name = "EleutherAI/pythia-6.9b"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
)

args = TrainingArguments(
    output_dir="./standard-finetune",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    bf16=True,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=500,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

RuntimeError: MPS backend out of memory (MPS allocated: 18.07 GB, other allocations: 384.00 KB, max allowed: 18.13 GB). Tried to allocate 96.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).